# Week 7 — Fine-Tuned Product Price Predictor (QLoRA)

# Goal:
# Use a fine-tuned Llama model to predict product prices and evaluate performance.

In [ ]:
# Install Dependencies
!pip install -q --upgrade bitsandbytes trl
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

# hese install:

#bitsandbytes → quantization for efficient models

#trl → training utilities

#util.py → evaluation helper

In [ ]:
# imports

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login

import torch
import transformers

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)

from datasets import load_dataset
from datetime import datetime
from peft import PeftModel

from util import evaluate

In [ ]:
# Project Constants
BASE_MODEL = "meta-llama/Llama-3.2-3B"

PROJECT_NAME = "price"
HF_USER = "remo4sam"

LITE_MODE = False

In [ ]:
# Dataset Configuration
DATA_USER = "ed-donner"

DATASET_NAME = (
    f"{DATA_USER}/items_prompts_lite"
    if LITE_MODE
    else f"{DATA_USER}/items_prompts_full"
)

In [ ]:
# Model Revision
if LITE_MODE:
    RUN_NAME = "2025-11-30_15.10.55-lite"
    REVISION = None
else:
    RUN_NAME = "2025-11-28_18.47.07"
    REVISION = "b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd"


PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"

HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

In [ ]:
# Quantization Configuration : This enables QLoRA 4-bit inference.
QUANT_4_BIT = True

capability = torch.cuda.get_device_capability()

use_bf16 = capability[0] >= 8

In [ ]:
# If you don't already have a HuggingFace account, visit https://huggingface.co to sign up and create a token.
# HuggingFace Login : Create a HuggingFace token and store it as a secret called HF_TOKEN.
hf_token = userdata.get("HF_TOKEN")

login(hf_token, add_to_git_credential=True)

In [ ]:
# Load Dataset
dataset = load_dataset(DATASET_NAME)

test = dataset["test"]

test[0]

In [ ]:
# Configure Quantization
if QUANT_4_BIT:

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4"
    )

else:

    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16
    )

In [ ]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# Load Base Model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
# Load Fine-Tuned Model
if REVISION:

    fine_tuned_model = PeftModel.from_pretrained(
        base_model,
        HUB_MODEL_NAME,
        revision=REVISION
    )

else:

    fine_tuned_model = PeftModel.from_pretrained(
        base_model,
        HUB_MODEL_NAME
    )


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
# Model Prediction Function
def model_predict(item):

    inputs = tokenizer(
        item["prompt"],
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():

        output_ids = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=8
        )

    prompt_len = inputs["input_ids"].shape[1]

    generated_ids = output_ids[0, prompt_len:]

    return tokenizer.decode(generated_ids)

In [ ]:
# Evaluate the Model
set_seed(42)

evaluate(model_predict, test)